# Data Cleaner and Preprocessing Notebook
This notebook demonstrates the step-by-step cleaning of raw scraped car data from Divar.
It contains all utility functions from `data_cleaner.py` split into separate cells for easy testing, followed by a Pandas cleaning demonstration.

### Imports and Logging Configuration

In [1]:
import os
import re
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)

### Merge raw cars files

In [2]:
def load_and_merge_cars(cars_dir='../data/cars', output_file='../data/raw_data.csv'):
    import glob
    import os
    import pandas as pd
    
    all_files = glob.glob(os.path.join(cars_dir, "*.csv"))
    
    if len(all_files) == 0:
        print("No files were found in the desired folder!")
        return pd.DataFrame()
        
    df_list = []
    for file in all_files:
        brand_name = os.path.basename(file).replace('.csv', '')
        
        temp_df = pd.read_csv(file)
        
        temp_df['brand'] = brand_name
        
        df_list.append(temp_df)

    raw_df = pd.concat(df_list, ignore_index=True)

    if 'brand_model' in raw_df.columns:
        raw_df.drop(columns=['brand_model'], inplace=True)

    raw_df.to_csv(output_file, index=False, encoding='utf-8-sig')

    print(f"Total Raw Ads Loaded: {len(raw_df)}")
    print(f"Raw data successfully saved to: {output_file}")
    
    return raw_df

In [3]:
raw_df = load_and_merge_cars()

Total Raw Ads Loaded: 2400
Raw data successfully saved to: ../data/raw_data.csv


### Copy Raw Data and Analysis Them

In [4]:
df = raw_df.copy()

In [5]:
df.drop(columns=['url'], inplace=True)

In [6]:
df.head()

,year,mileage,price,insurance,gearbox_type,gearbox_health,motor_status,body_status,chassis_status,brand
0,۲۰۱۵,۱۸۰۰۰۰,"۵,۸۵۰,۰۰۰,۰۰۰ تومان",NaN,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45
1,۲۰۱۵,۱۳۰۰۰۰,"۴,۸۹۰,۰۰۰,۰۰۰ تومان",۶ ماه,اتوماتیک,سالم و پلمپ,سالم,رنگ‌شدگی در ۱ ناحیه,سالم و پلمپ,hyundai_santafe-ix45
2,۲۰۱۲,۱۹۰۰۰۰,"۶,۵۰۰,۰۰۰,۰۰۰ تومان",NaN,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45
3,۲۰۰۸,۲۶۵۰۰۰,"۴,۲۵۰,۰۰۰,۰۰۰ تومان",۱۲ ماه,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45
4,۲۰۱۰,۲۷۰۰۰۰,"۵,۰۵۰,۰۰۰,۰۰۰ تومان",۱۲ ماه,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2400 entries, 0 to 2399
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   year            2331 non-null   object
 1   mileage         2331 non-null   object
 2   price           2330 non-null   object
 3   insurance       1991 non-null   object
 4   gearbox_type    2235 non-null   object
 5   gearbox_health  2082 non-null   object
 6   motor_status    2330 non-null   object
 7   body_status     2330 non-null   object
 8   chassis_status  2330 non-null   object
 9   brand           2400 non-null   object
dtypes: object(10)
memory usage: 187.6+ KB


In [8]:
df.duplicated().sum()

65

In [9]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Total Unique Ads after dropping duplicates: {len(df)}")

Total Unique Ads after dropping duplicates: 2335


### Clearing Functions

In [10]:
def persian_to_english_digits(text):
    if not isinstance(text, str): return ""
    return text.translate(str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789"))

def clean_year(year_str):
    normalized = persian_to_english_digits(str(year_str))
    years = [int(y) for y in re.findall(r"\b\d{4}\b", normalized)]
    if not years: 
        return np.nan
    for y in years:
        if y > 1900:
            return y
    return years[0] + 621

def clean_mileage(mileage_str):
    normalized = persian_to_english_digits(str(mileage_str)).strip().lower()
    if normalized == "0" or any(kw in normalized for kw in ["صفر", "کارتکس", "کارنکرده", "نو"]):
        return 0
    digits = re.sub(r"\D", "", normalized)
    return int(digits) if digits else 0

def clean_seller_price(price_str):
    normalized = persian_to_english_digits(str(price_str)).strip()
    if "توافقی" in normalized or "agreement" in normalized.lower():
        return np.nan
    digits = re.sub(r"\D", "", normalized)
    return int(digits) if digits else np.nan

def clean_insurance(insurance_str):
    normalized = persian_to_english_digits(str(insurance_str)).strip().lower()
    if any(kw in normalized for kw in ["بدون", "فاقد", "no"]): return 0
    digits = re.sub(r"\D", "", normalized)
    return int(digits) if digits else 0


In [11]:
df['year'] = df['year'].apply(clean_year)
df['mileage'] = df['mileage'].apply(clean_mileage)
df['price'] = df['price'].apply(clean_seller_price)
df['insurance'] = df['insurance'].apply(clean_insurance)
df.head()

,year,mileage,price,insurance,gearbox_type,gearbox_health,motor_status,body_status,chassis_status,brand
0,2015.0,180000,5.850000e+09,0,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45
1,2015.0,130000,4.890000e+09,6,اتوماتیک,سالم و پلمپ,سالم,رنگ‌شدگی در ۱ ناحیه,سالم و پلمپ,hyundai_santafe-ix45
2,2012.0,190000,6.500000e+09,0,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45
3,2008.0,265000,4.250000e+09,12,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45
4,2010.0,270000,5.050000e+09,12,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45


### Check Unique Values

In [12]:
def check_unique_values(df):
    print("Checking for unique values for each column:")
    print("-" * 50)
    
    for col in df.columns:
        num_unique = df[col].nunique(dropna=False)
        
        if num_unique <= 20:
            unique_vals = df[col].unique()
            print(f"🔹 {col}: {num_unique} values -> {unique_vals}")
        else:
            print(f"🔸 {col}: {num_unique} values (Too many to list)")


In [13]:
check_unique_values(df)

Checking for unique values for each column:
--------------------------------------------------
🔸 year: 31 values (Too many to list)
🔸 mileage: 486 values (Too many to list)
🔸 price: 682 values (Too many to list)
🔹 insurance: 13 values -> [ 0  6 12 10  8  9  3 11  5  7  4  2  1]
🔹 gearbox_type: 3 values -> ['اتوماتیک' nan 'دنده\u200cای']
🔹 gearbox_health: 5 values -> ['سالم و پلمپ' nan 'تعمیر شده' 'نیاز به تعمیر جزئی' 'نیاز به تعمیر اساسی']
🔹 motor_status: 5 values -> ['سالم' 'تعویض شده' 'تعیین\u200cنشده' nan 'نیاز به تعمیر']
🔹 body_status: 14 values -> ['سالم و بی\u200cخط و خش' 'رنگ\u200cشدگی در ۱ ناحیه'
 'رنگ\u200cشدگی در ۲ ناحیه' 'خط و خش جزیی' nan 'رنگ\u200cشدگی در ۳ ناحیه'
 'رنگ\u200cشدگی در ۴ ناحیه' 'دوررنگ' 'تصادفی' 'صافکاری بی\u200cرنگ'
 'تمام\u200cرنگ' 'اوراقی' 'رنگ\u200cشدگی در ۵ ناحیه'
 'رنگ\u200cشدگی در ۷ ناحیه']
🔹 chassis_status: 5 values -> ['سالم و پلمپ' nan 'تعیین\u200cنشده' 'ضربه\u200cخورده' 'رنگ\u200cشده']
🔹 brand: 10 values -> ['hyundai_santafe-ix45' 'mvm_x22' 'dena' 

### Managing Hidden Missing Data

In [14]:
def detect_hidden_missing_data(df, suspicious_values=None, target_column=None):
    """
    Detects missing data in the DataFrame for all columns or a specific column,
    including standard NaN/None and custom suspicious values.

    Parameters:
    - df: DataFrame containing the data.
    - suspicious_values: List of custom values to treat as missing (e.g., [-999, 'unknown', '']).
                         Defaults to None (only checks NaN/None).
    - target_column: String, name of a specific column to check for missing data.
                    If None, checks all columns. Defaults to None.

    Returns:
    - A DataFrame summarizing missing counts and percentages for the selected column(s).
    """
    if suspicious_values is None:
        suspicious_values = []

    # Create a copy to avoid modifying the original
    temp_df = df.copy()

    # Validate target_column
    if target_column is not None:
        if target_column not in temp_df.columns:
            print(f"Error: Column '{target_column}' not found in DataFrame.")
            return pd.DataFrame()
        temp_df = temp_df[[target_column]]
        print(f"Checking missing data for column: {target_column}")
    else:
        print("Checking missing data for all columns")

    # Replace suspicious values with NaN
    for val in suspicious_values:
        temp_df = temp_df.replace(val, np.nan)

    # Calculate missing counts and percentages
    missing_counts = temp_df.isnull().sum()
    missing_percentages = (missing_counts / len(temp_df)) * 100
    missing_summary = pd.DataFrame({
        'Missing Count': missing_counts,
        'Missing Percentage (%)': missing_percentages.round(2)
    }).sort_values(by='Missing Count', ascending=False)

    # Filter summary to show only columns with missing data
    missing_summary = missing_summary[missing_summary['Missing Count'] > 0]

    # Print summary
    if not missing_summary.empty:
        print("Missing Data Summary:")
        print(missing_summary)
    else:
        print("No missing data detected in the selected column(s).")

    return missing_summary

In [15]:
suspicious = ['تعیین‌نشده', 'تعیین نشده', '', '-']
missing_summary = detect_hidden_missing_data(
    df, 
    suspicious_values=suspicious
)


Checking missing data for all columns
Missing Data Summary:
                Missing Count  Missing Percentage (%)
gearbox_health            258                   11.05
chassis_status            174                    7.45
motor_status              106                    4.54
gearbox_type              105                    4.50
price                      11                    0.47
body_status                11                    0.47
year                       10                    0.43


In [16]:
to_replace = ['تعیین‌نشده', 'تعیین نشده', '', '-']
df.replace(to_replace, np.nan, inplace=True)

### Show Unique Values and Percentages of Missing Values

In [17]:
def missing_and_unique_info(df):
    """
    Returns a report of columns with missing values, including:
    - Missing percentage
    - Number of unique values (only for object/categorical columns)

    Parameters:
        df (pd.DataFrame): Input DataFrame

    Returns:
        pd.DataFrame: Sorted report of columns with missing data
    """
    # Calculate missing stats
    missing_count = df.isnull().sum()
    missing_percent = round((missing_count / len(df) * 100), 2)

    # Filter columns with missing values
    cols_with_missing = missing_count[missing_count > 0].index

    if len(cols_with_missing) == 0:
        print("No columns with missing values.")
        return pd.DataFrame()

    # Count missing values
    missing_count = df[cols_with_missing].isnull().sum()

    # Count unique values only for object/categorical columns
    unique_counts = pd.Series(index=cols_with_missing, dtype='object')
    for col in cols_with_missing:
        if df[col].dtype == 'object' or  isinstance(df[col].dtype, pd.CategoricalDtype):
            unique_counts[col] = df[col].nunique(dropna=True)
        else:
            unique_counts[col] = np.nan

    # Build report
    report = pd.DataFrame({
        'Missing %': missing_percent[cols_with_missing],
        'Total Count': len(df),
        'Missing Count': missing_count,
        'Unique Values': unique_counts
    }).sort_values(by='Missing %', ascending=False)

    return report

In [18]:
result_df = missing_and_unique_info(df)
print(result_df)

                Missing %  Total Count  Missing Count Unique Values
gearbox_health      11.05         2335            258             4
chassis_status       7.45         2335            174             3
motor_status         4.54         2335            106             3
gearbox_type         4.50         2335            105             2
price                0.47         2335             11           NaN
body_status          0.47         2335             11            13
year                 0.43         2335             10           NaN


In [19]:
df.dropna(subset=['price', 'year', 'body_status'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Data shape after handling missing values: {df.shape}")

Data shape after handling missing values: (2324, 10)


In [20]:
result_df = missing_and_unique_info(df)
print(result_df)

                Missing %  Total Count  Missing Count Unique Values
gearbox_health      10.63         2324            247             4
chassis_status       7.01         2324            163             3
motor_status         4.09         2324             95             3
gearbox_type         4.04         2324             94             2


### Feature Engineering

In [21]:
def create_features(df):
    df = df.copy()
    
    df['car_age'] = 2026 - df['year']
    df['car_age'] = df['car_age'].apply(lambda x: 0 if x < 0 else x)
    
    df['yearly_mileage'] = df['mileage'] / (df['car_age'] + 1)
    
    df['has_insurance'] = df['insurance'].apply(lambda x: 1 if x > 0 else 0)
    
    return df

In [22]:
df = create_features(df)

In [23]:
df.head()

,year,mileage,price,insurance,gearbox_type,gearbox_health,motor_status,body_status,chassis_status,brand,car_age,yearly_mileage,has_insurance
0,2015.0,180000,5.850000e+09,0,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45,11.0,15000.000000,0
1,2015.0,130000,4.890000e+09,6,اتوماتیک,سالم و پلمپ,سالم,رنگ‌شدگی در ۱ ناحیه,سالم و پلمپ,hyundai_santafe-ix45,11.0,10833.333333,1
2,2012.0,190000,6.500000e+09,0,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45,14.0,12666.666667,0
3,2008.0,265000,4.250000e+09,12,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45,18.0,13947.368421,1
4,2010.0,270000,5.050000e+09,12,اتوماتیک,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,hyundai_santafe-ix45,16.0,15882.352941,1


In [24]:
df.describe()

,year,mileage,price,insurance,car_age,yearly_mileage,has_insurance
count,2324.000000,2324.000000,2.324000e+03,2324.000000,2324.000000,2324.000000,2324.000000
mean,2016.904045,146020.516351,1.066270e+10,6.393718,9.095955,13522.064647,0.854561
std,5.586004,115959.736701,1.907953e+11,3.911458,5.586004,7171.650116,0.352619
min,1993.000000,0.000000,1.000000e+03,0.000000,0.000000,0.000000,0.000000
25%,2014.000000,58000.000000,8.900000e+08,3.000000,4.000000,9250.000000,1.000000
50%,2017.000000,120000.000000,1.550000e+09,6.000000,9.000000,13637.626263,1.000000
75%,2022.000000,207000.000000,2.570000e+09,10.000000,12.000000,17647.058824,1.000000
max,2026.000000,1000000.000000,6.300000e+12,12.000000,33.000000,100000.000000,1.000000


In [25]:
df.describe(include='object')

,gearbox_type,gearbox_health,motor_status,body_status,chassis_status,brand
count,2230,2077,2229,2324,2161,2324
unique,2,4,3,13,3,10
top,دنده‌ای,سالم و پلمپ,سالم,سالم و بی‌خط و خش,سالم و پلمپ,mvm_x22
freq,1439,2055,2202,1502,2096,237


In [26]:
df.isna().sum()

year                0
mileage             0
price               0
insurance           0
gearbox_type       94
gearbox_health    247
motor_status       95
body_status         0
chassis_status    163
brand               0
car_age             0
yearly_mileage      0
has_insurance       0
dtype: int64

### Split Data and Save them

In [27]:
X = df.drop(columns=['price'])
y = df['price']

In [28]:
from sklearn.model_selection import  train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50,
    random_state=42
)

In [29]:
print(f"Train data size: {X_train.shape[0]} (80%)")
print(f"Validation data size: {X_val.shape[0]} (10%)")
print(f"Test data size: {X_test.shape[0]} (10%)")

Train data size: 1859 (80%)
Validation data size: 232 (10%)
Test data size: 233 (10%)


### Categorical Imputation

In [30]:
def fill_categorical_splits(X_train, X_val, X_test, cat_cols):
    """
    Fill in the empty values ​​of the category using the most frequent mode.
    To avoid data leakage, the mode is only calculated on X_train.
    """
    X_train_c = X_train.copy()
    X_val_c = X_val.copy()
    X_test_c = X_test.copy()
    
    for col in cat_cols:
        train_mode = X_train_c[col].mode()[0]
        
        X_train_c[col] = X_train_c[col].fillna(train_mode)
        X_val_c[col] = X_val_c[col].fillna(train_mode)
        X_test_c[col] = X_test_c[col].fillna(train_mode)
        
    return X_train_c, X_val_c, X_test_c

In [31]:
cols_to_fill = ['gearbox_health', 'chassis_status', 'motor_status', 'gearbox_type']

X_train, X_val, X_test = fill_categorical_splits(X_train, X_val, X_test, cat_cols=cols_to_fill)

In [32]:
result_train = missing_and_unique_info(X_train)
result_val = missing_and_unique_info(X_val)
result_test = missing_and_unique_info(X_test)

print(result_train)
print(result_test)
print(result_val)

No columns with missing values.
No columns with missing values.
No columns with missing values.
Empty DataFrame
Columns: []
Index: []
Empty DataFrame
Columns: []
Index: []
Empty DataFrame
Columns: []
Index: []


### Saving Cleaning Data

In [ ]:
X_train.join(y_train).to_csv('../data/train_clean.csv', index=False, encoding='utf-8-sig')
X_val.join(y_val).to_csv('../data/val_clean.csv', index=False, encoding='utf-8-sig')
X_test.join(y_test).to_csv('../data/test_clean.csv', index=False, encoding='utf-8-sig')

print("Cleaned datasets have been successfully saved!")

Cleaned datasets have been successfully saved!
